In [124]:
# !pip install openpyxl

In [125]:
import random
import string
from datetime import datetime, timedelta
import pandas as pd
import re


In [126]:
def gerar_cpf():
    """
    Gera CPF formatado.
    Exemplo: 123.456.789-01
    """
    nums = [random.randint(0, 9) for _ in range(11)]

    return (
        f"{nums[0]}{nums[1]}{nums[2]}."
        f"{nums[3]}{nums[4]}{nums[5]}."
        f"{nums[6]}{nums[7]}{nums[8]}-"
        f"{nums[9]}{nums[10]}"
    )

In [127]:
def gerar_cnpj():
    """
    Gera CNPJ formatado.
    Exemplo: 12.345.678/0001-99
    """
    nums = [random.randint(0, 9) for _ in range(14)]

    return (
        f"{nums[0]}{nums[1]}."
        f"{nums[2]}{nums[3]}{nums[4]}."
        f"{nums[5]}{nums[6]}{nums[7]}/"
        f"{nums[8]}{nums[9]}{nums[10]}{nums[11]}-"
        f"{nums[12]}{nums[13]}"
    )

In [128]:
def gerar_dados_sinteticos(qtd_registros=100):
    """
    Gera dados sintéticos de chamados.

    Campos:
    - id_task
    - cd_pin
    - cd_modelo
    - nr_contrato
    - cd_unico_cliente
    - nr_cpf_cnpj
    - nr_ano_mes
    """

    modelos = [
        "9492", "90", "511",
        "201", "946", "251"
    ]

    pins = [
          "2050", "2051", "3000", 
          "3050", "5032", "3054"]

    seguimentos = [
          "3", "9", "256", "11"
    ]


    dados = []

    for i in range(qtd_registros):

        # Decide aleatoriamente se será CPF ou CNPJ
        if random.choice([True, False]):
                nr_cpf_cnpj = gerar_cpf()
        else:
            nr_cpf_cnpj = gerar_cnpj()

        # Gera ano/mês aleatório entre 2020 e hoje
        data_inicio = datetime(2026, 2, 1)
        data_fim = datetime.now()

        dias_range = (data_fim - data_inicio).days
        data_random = data_inicio + timedelta(days=random.randint(0, dias_range))

        nr_ano_mes = data_random.strftime("%Y%m")

        registro = {
            "ID_TASK": i + 9435,
            "CD_PIN": random.choice(pins),
            "CD_MODELO": random.choice(modelos),
            "NR_CONTRATO": f"66000{random.randint(1000000, 9999999)}",
            "CD_UNIDADE": random.randint(10000000, 99999999),
            "CD_SEGMENTO": random.choice(seguimentos),
            "CPF_CNPJ": nr_cpf_cnpj,
            "ANOMES": nr_ano_mes
        }

        dados.append(registro)

    return dados

In [129]:
def aplicar_ruido_nos_dados(lista_dados, percentual_ruido=0.20):
    """Consome a lista de dados e aplica ruídos/inversões de forma INDEPENDENTE

    nos registros, baseando-se no percentual escolhido.
    """
    bancos = ["033"]

    for registro in lista_dados:

        # --- ERRO 1: CONTRATO RECEBE BANCO + AGÊNCIA ---
        # Acontece de forma independente para a % escolhida
        if random.random() < percentual_ruido:
            cod_banco = random.choice(bancos)
            cod_agencia = f"{random.randint(1, 9999):04d}"
            contrato_original = registro["NR_CONTRATO"]
            registro["NR_CONTRATO"] = (
                f"{cod_banco}{cod_agencia}{contrato_original}"
            )

        # --- ERRO 2: UNIDADE (PERNUMPER) FICA SUJA ---
        # Também independente. Pode acontecer junto com o erro acima, ou sozinho
        if random.random() < percentual_ruido:
            chance_sujeira = random.random()
            if chance_sujeira < 0.50:  # 50% de chance de virar 0, 1, 0000
                registro["CD_UNIDADE"] = random.choice(
                    ["0", "1", "0000", "00000000"]
                )
            else:  # 50% de chance de virar texto alfanumérico misto
                tamanho = random.randint(4, 10)
                registro["CD_UNIDADE"] = "".join(
                    random.choice(string.ascii_uppercase + string.digits)
                    for _ in range(tamanho)
                )

        # --- ERRO 3: INVERSÃO DE COLUNAS ---
        # Troca os valores de lugar de forma independente
        if random.random() < percentual_ruido:
            registro["NR_CONTRATO"], registro["CD_UNIDADE"] = (
                registro["CD_UNIDADE"],
                registro["NR_CONTRATO"],
            )

    return lista_dados

In [130]:
def motorCorpao():
    dfGrade = pd.read_excel("./database/gold/gradePesos.xlsx")
    dfModelo = pd.read_excel("./database/gold/modeloAngariacao.xlsx")
    dfChamados = pd.read_excel("./database/Bronze/chamados.xlsx")

    dfChamados["NR_CONTRATO"]


In [131]:
def capturarContrato(x):
    # Encontra todas as ocorrências
    contrato = re.findall(r'66\d+', x)
    
    # Se a lista não estiver vazia, retorna o primeiro item [0]
    # Se estiver vazia, retorna None ou uma string vazia
    return contrato[0] if contrato else None

In [ ]:
dfChamados = pd.read_excel("./database/Bronze/chamados.xlsx")

dfChamados["regexContrato"] = dfChamados["NR_CONTRATO"].apply(lambda x: capturarContrato(str(x)))


0      False
1       True
2      False
3       True
4      False
       ...  
495    False
496     True
497    False
498    False
499    False
Name: regexContrato, Length: 500, dtype: bool

In [143]:
dfChamados.loc[~dfChamados["regexContrato"].isna(), ["regexContrato"]]

,regexContrato
0,660009654089
2,660007137616
4,660009151360
5,660001178044
7,660003844071
...,...
493,660005581208
495,660009810173
497,660001221335
498,660002279704


In [134]:
# 1. Gera dados limpos
dados_originais = gerar_dados_sinteticos(500)

# 2. Aplica ruído
dados_com_ruido = aplicar_ruido_nos_dados(dados_originais, percentual_ruido=0.2)
df = pd.DataFrame(dados_com_ruido)
df.to_excel("./database/Bronze/chamados.xlsx")